# Reproduce the baseline on the real ADC2021 data

This notebook downloads the public dataset from Zenodo (about 800 MB), trains the
baseline autoencoder on Standard Model background events and measures how well it
separates the four new-physics signals. It needs no GPU and takes about ten minutes
on a free Colab machine.

Run all cells from top to bottom (Runtime > Run all).

## 1. Get the code

In [ ]:
![ -d lhc-anomaly-detection ] || git clone -q https://github.com/berkeekicii/lhc-anomaly-detection.git
%cd lhc-anomaly-detection
!python --version

## 2. Run the automated tests

They use small fake files in the dataset's format, so they finish in under a minute.

In [ ]:
!python -m pytest -q -p no:cacheprovider

## 3. Download the dataset

Five HDF5 files from Zenodo; every file is checked against its MD5 checksum.

In [ ]:
!python -m lhcad.download

## 4. Look inside the files

Structure, one example event and object statistics for the background and for each signal.

In [ ]:
from lhcad.data import find_raw_files

for name, path in find_raw_files("data/raw").items():
    print("=" * 30, name, "=" * 30)
    !python -m lhcad.explore {path}

## 5. Prepare the arrays

Keeps (pT, eta, phi) of the 19 objects, shuffles the background and splits it 50% / 10% / 40% into train / validation / test.

In [ ]:
!python -m lhcad.prepare

## 6. Train the baseline autoencoder

In [ ]:
!python -m lhcad.train --name baseline

## 7. Evaluate on the four signals

AUC and the true-positive rate at false-positive rates of 1e-3, 1e-4 and 1e-5 (the challenge metric).

In [ ]:
!python -m lhcad.evaluate --run runs/baseline

In [ ]:
from IPython.display import Image, display

for name in ("loss.png", "roc.png", "scores.png"):
    display(Image(f"runs/baseline/{name}", width=700))

## 8. The numbers behind the table

In [ ]:
for path in ("data/processed/meta.json", "runs/baseline/train_config.json", "runs/baseline/metrics.json"):
    print(path)
    print(open(path).read())